# Linear Regression

## Overview

OLS linear regression models a continuous outcome as a linear function of one or more predictors. The slope is the expected change in y per unit increase in x, holding all else constant.

**Key outputs:**

| Output | What it tells you |
|---|---|
| Coefficients | Expected change in y per unit x |
| R-squared | Proportion of variance explained |
| Residual SE | Average prediction error (in y units) |
| p-value / CI | Evidence and uncertainty for each coefficient |

Fit with `statsmodels` for inference (CIs, p-values, diagnostics); use `sklearn` for prediction pipelines.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

rng = np.random.default_rng(42)
n = 120
elevation = rng.uniform(50, 400, n)
richness  = 30 - 0.04*elevation + rng.normal(0, 4, n)

df = pd.DataFrame({"elevation": elevation, "richness": richness})
print(f"Correlation: r={df.corr().iloc[0,1]:.3f}")

---
## Fitting with statsmodels

In [ ]:
X = sm.add_constant(df["elevation"])
model = sm.OLS(df["richness"], X).fit()
print(model.summary())
print(f"\nInterpretation: each 1m increase in elevation associated with")
print(f"  {model.params[1]:.4f} species change (95% CI: {model.conf_int().iloc[1,0]:.4f} to {model.conf_int().iloc[1,1]:.4f})")

---
## Visualising the Fit

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
# Scatter + regression line
x_range = np.linspace(elevation.min(), elevation.max(), 200)
X_pred = sm.add_constant(x_range)
y_pred = model.predict(X_pred)
pred_ci = model.get_prediction(X_pred).conf_int(alpha=0.05)
axes[0].scatter(elevation, richness, alpha=0.4, s=20, color="steelblue")
axes[0].plot(x_range, y_pred, color="#e74c3c", lw=2, label="Fitted")
axes[0].fill_between(x_range, pred_ci[:,0], pred_ci[:,1], alpha=0.15, color="#e74c3c", label="95% CI")
axes[0].set_xlabel("Elevation (m)"); axes[0].set_ylabel("Richness")
axes[0].set_title(f"OLS: slope={model.params[1]:.4f}, R2={model.rsquared:.3f}")
axes[0].legend()
# Residuals vs fitted
resid = model.resid; fitted = model.fittedvalues
axes[1].scatter(fitted, resid, alpha=0.4, s=20, color="steelblue")
axes[1].axhline(0, color="#e74c3c", lw=1.5)
axes[1].set_xlabel("Fitted values"); axes[1].set_ylabel("Residuals")
axes[1].set_title("Residuals vs Fitted")
plt.tight_layout(); plt.show()

---
## Prediction Intervals vs Confidence Intervals

In [ ]:
# CI on the mean: uncertainty about the regression line
# PI on a new observation: uncertainty about a single new data point (wider)
pred_frame = pd.DataFrame({"elevation": [100, 200, 300]})
X_new = sm.add_constant(pred_frame["elevation"])
pred = model.get_prediction(X_new)
print(pred.summary_frame(alpha=0.05)[["mean","mean_ci_lower","mean_ci_upper","obs_ci_lower","obs_ci_upper"]])
print("\nmean_ci: CI on the mean response (line uncertainty)")
print("obs_ci:  PI for a new individual observation (always wider)")

---
## Checking Assumptions

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
# Q-Q plot of residuals
(osm,osr),(sl,ic,r) = stats.probplot(model.resid, dist="norm")
axes[0].scatter(osm, osr, s=15, alpha=0.6, color="steelblue")
axes[0].plot(osm, sl*np.array(osm)+ic, color="#e74c3c", lw=1.5)
axes[0].set_title(f"Q-Q Residuals (r={r:.3f})")
axes[0].set_xlabel("Theoretical"); axes[0].set_ylabel("Sample")
# Scale-location (sqrt |residuals| vs fitted)
axes[1].scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.4, s=20, color="steelblue")
axes[1].set_xlabel("Fitted"); axes[1].set_ylabel("sqrt(|residuals|)")
axes[1].set_title("Scale-Location (heteroscedasticity check)")
plt.tight_layout(); plt.show()
from statsmodels.stats.diagnostic import het_breuschpagan
lm, lm_p, f, f_p = het_breuschpagan(model.resid, model.model.exog)
print(f"Breusch-Pagan heteroscedasticity test: p={lm_p:.4f}")

---

## Common Pitfalls

**1. Interpreting R-squared as a measure of model correctness**  
R-squared measures the proportion of variance explained, not whether the model is correctly specified. A high R-squared with a non-linear true relationship still produces biased estimates. Always inspect residual plots.

**2. Confusing prediction intervals and confidence intervals**  
The CI on the mean response captures uncertainty about the regression line. A prediction interval for a new observation is always wider because it includes both line uncertainty and individual variability. Never use a CI to make a statement about where a new observation will fall.

**3. Extrapolating beyond the range of observed x values**  
Linear relationships observed within a data range may not hold outside it. Predictions at extreme x values should be reported with explicit warnings about extrapolation.

**4. Not checking residual plots**  
OLS assumption violations (non-linearity, heteroscedasticity, outliers) are most clearly visible in residual vs fitted and scale-location plots, not in model summaries. Always plot before interpreting coefficients.

**5. Reporting coefficients without confidence intervals**  
A coefficient with p<0.05 but a CI spanning nearly zero to large values is poorly estimated. Always report CIs alongside point estimates so readers can judge precision.
---
*python_methods_library - Samantha McGarrigle*